# Logistic Regression and Model Evaluation – Avalanche Accidents in Switzerland

**Dataset:** All known avalanche accidents in Switzerland with at least one person caught, recorded by the WSL Institute
for Snow and Avalanche Research SLF (EnviDat, since 1970/71). We use the hydrological years 2000/01–2023/24.
File: `../Data/avalanche_accidents_switzerland.csv`.

| variable | meaning |
|---|---|
| `date`, `hydrological_year`, `month` | date of the accident (a hydrological year runs from October to September) |
| `canton`, `municipality`, `latitude`, `longitude` | location of the start zone |
| `elevation`, `aspect`, `inclination` | elevation (m a.s.l.), slope aspect (N, NE, …) and inclination (°) of the start zone |
| `danger_level` | avalanche danger level of the bulletin (1 = low … 5 = very high) |
| `activity` | `tour` (backcountry touring), `offpiste` (off-piste skiing), `transportation.corridor` (roads, railways), `building` |
| `caught`, `fully_buried`, `dead` | number of persons caught, completely buried and killed |
| `fatal` | 1 = at least one person died – **target** |

**Business question:** When an avalanche alarm comes in, the rescue coordination has to decide within minutes which
resources to send (helicopter with emergency doctor, avalanche dogs, …). How well can the probability of a **fatal**
outcome be estimated from the information available at that moment, and which **threshold** should trigger the maximum
alarm level?

**Slides:** Part 04 «Classification models» – logistic regression, evaluation (confusion matrix, metrics, thresholds,
ROC curve and AUC).

**After this exercise you can ...**
- compute probabilities of a logistic regression model by hand,
- fit and interpret a logistic regression in scikit-learn (including categorical features),
- compute and interpret accuracy, precision, recall, F1 and FPR,
- choose a threshold based on the costs of false positives and false negatives,
- draw and interpret a ROC curve and the AUC,
- explain why the choice of features must match the information available at prediction time.

## Concept

The concepts behind this exercise (formulas, worked examples of the slides and additional explanations) are
explained in the notebook [logistic_regression_evaluation_concept.ipynb](logistic_regression_evaluation_concept.ipynb). Read it before you start.

## Libraries and settings

In [ ]:
# Libraries
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (confusion_matrix, accuracy_score, precision_score, recall_score,
                             f1_score, roc_curve, roc_auc_score)

# Ignore warnings
import warnings
warnings.filterwarnings('ignore')

# Show current working directory
print(os.getcwd())

## Exercise 1: Calculations by hand

**a) Probabilities.** A logistic regression with the only feature *number of completely buried persons* has the
coefficients $\beta_0 = -3.5$ and $\beta_1 = 2.5$. Compute the probability of a fatal outcome for accidents with 0, 1
and 2 completely buried persons. Which accidents are classified as fatal with $\tau = 0.5$ and with $\tau = 0.2$?

**b) Metrics.** A model is applied to 1'000 accidents, 130 of which were fatal. It flags 160 accidents as fatal, of
which 100 really were. Fill in the confusion matrix and compute accuracy, precision, recall, FPR and F1.

**c) Baseline.** What accuracy does a "model" reach that simply predicts "not fatal" for every accident? What is its
recall?

**d)** Verify your results with Python.

In [ ]:
# a) Probabilities
beta_0, beta_1 = -3.5, 2.5
buried = np.array([0, 1, 2])
score = beta_0 + beta_1 * buried
p = 1 / (1 + np.exp(-score))
for n, s, pi in zip(buried, score, p):
    print(f'fully buried = {n}: score = {s:5.2f}, p = {pi:.3f}, fatal (tau=0.5): {pi >= 0.5}, fatal (tau=0.2): {pi >= 0.2}')

# b) Metrics
TP, FP, FN, TN = 100, 60, 30, 810
n = TP + FP + FN + TN
precision = TP / (TP + FP)
recall = TP / (TP + FN)
print(f'\nAccuracy  = {(TP + TN) / n:.3f}')
print(f'Precision = {precision:.3f}')
print(f'Recall    = {recall:.3f}')
print(f'FPR       = {FP / (FP + TN):.3f}')
print(f'F1        = {2 * precision * recall / (precision + recall):.3f}')

# c) Baseline: always predict 'not fatal'
print(f'\nBaseline accuracy = {(FP + TN) / n:.3f}, baseline recall = 0')

**Solution:**

a) Scores −3.5, −1.0, 1.5 → $p = 1/(1+e^{3.5}) \approx 0.029$, $p = 1/(1+e^{1}) \approx 0.269$,
$p = 1/(1+e^{-1.5}) \approx 0.818$. With $\tau = 0.5$ only the accident with 2 buried persons is classified as fatal;
with $\tau = 0.2$ also the accident with 1 buried person.
(Optional: $e^{2.5} \approx 12$ → every additional buried person multiplies the odds of a fatal outcome by about 12.)

b) TP = 100, FP = 160 − 100 = 60, FN = 130 − 100 = 30, TN = 1000 − 100 − 60 − 30 = 810.
Accuracy = 910/1000 = 0.91; Precision = 100/160 = 0.625; Recall = 100/130 ≈ 0.769; FPR = 60/870 ≈ 0.069;
F1 = 2·0.625·0.769/(0.625+0.769) ≈ 0.690.

c) Always predicting "not fatal" gives an accuracy of 870/1000 = **0.87** – almost as high as the model – but a recall of
**0**: not a single fatal accident is recognised. With imbalanced classes, accuracy alone misleads.

## Exercise 2: Import and explore the data

a) Import the data and keep the hydrological years 2000/01 to 2023/24 (`hydrological_year >= '2000/01'`). How many
accidents remain? What share of them was fatal?

b) Plot the share of fatal accidents by `danger_level`, by `activity` and by the number of completely buried persons
(`fully_buried`, grouped into 0, 1, 2, 3+). Which variable separates fatal and non-fatal accidents best?

In [ ]:
# a) Import the data
aval = pd.read_csv('../Data/avalanche_accidents_switzerland.csv', sep=',')
aval = aval[aval['hydrological_year'] >= '2000/01'].reset_index(drop=True)
print('Accidents 2000/01-2023/24:', len(aval))
print('Share of fatal accidents:', round(aval['fatal'].mean(), 3))
print('Missing values (share):')
print(aval[['danger_level', 'elevation', 'aspect', 'activity', 'caught', 'fully_buried']].isna().mean().round(2))

# b) Share of fatal accidents by danger level, activity and number of buried persons
aval['buried_group'] = pd.cut(aval['fully_buried'], [-1, 0, 1, 2, 100], labels=['0', '1', '2', '3+'])
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8), sharey=True)
for ax, col in zip(axes, ['danger_level', 'activity', 'buried_group']):
    rate = aval.groupby(col, observed=True)['fatal'].agg(['mean', 'size'])
    ax.bar(rate.index.astype(str), rate['mean'], color='steelblue')
    for i, (m, s) in enumerate(zip(rate['mean'], rate['size'])):
        ax.text(i, m + 0.01, f'{m:.0%}\n(n={s})', ha='center', fontsize=8)
    ax.set_title(f'Share of fatal accidents by {col}')
    ax.tick_params(axis='x', rotation=20)
axes[0].set_ylabel('Share fatal')
axes[0].set_ylim(0, 1.05)
plt.show()

**Solution:**

a) 2'876 accidents remain; **13.2 %** of them were fatal → the classes are imbalanced. The danger level is missing for
16 % of the accidents, elevation and aspect for 6–7 %.

b) The share of fatal accidents hardly differs between **danger levels** (≈ 11 % at levels 1–2, 12 % at level 3, 16 % at
level 4) or **activities** (tour 14 %, off-piste 13 %, transportation corridor 10 %). The **number of completely buried
persons** separates much better: if nobody was completely buried, only **2 %** of the accidents were fatal; with one
buried person 43 %, with two 52 %, with three or more 72 %.

Important: these data contain only **accidents**. A higher danger level clearly increases the *risk of an accident*,
but given that an accident has happened, it hardly changes the probability that it is fatal.

## Exercise 3: Logistic regression with one feature

a) Fit a logistic regression `fatal ~ fully_buried` on **all** accidents of Exercise 2 (no split yet). Print
$\hat\beta_0$ and $\hat\beta_1$ and compare them with the coefficients of Exercise 1a.

b) Plot the predicted probability against `fully_buried` (0–8) together with the observed share of fatal accidents per
value (dots). Interpret the S-curve.

In [ ]:
# a) Logistic regression with fully_buried only
log_reg_1 = LogisticRegression()
log_reg_1.fit(aval[['fully_buried']], aval['fatal'])
print('beta_0 =', round(log_reg_1.intercept_[0], 3), ' beta_1 =', round(log_reg_1.coef_[0][0], 3))

# b) Plot predicted probability and observed share
grid = pd.DataFrame({'fully_buried': np.linspace(0, 8, 200)})
observed = aval[aval['fully_buried'] <= 8].groupby('fully_buried')['fatal'].agg(['mean', 'size'])

plt.figure(figsize=(7, 4))
plt.scatter(observed.index, observed['mean'], s=np.sqrt(observed['size']) * 8, color='grey',
            label='observed share (dot size ~ number of accidents)')
plt.plot(grid['fully_buried'], log_reg_1.predict_proba(grid)[:, 1], color='darkred', linewidth=2,
         label='logistic regression')
plt.axhline(0.5, color='black', linestyle=':', linewidth=1, label='threshold 0.5')
plt.xlabel('Number of completely buried persons')
plt.ylabel('P(fatal)')
plt.ylim(0, 1.05)
plt.legend(fontsize=8)
plt.grid(alpha=0.3)
plt.show()

**Solution:**

a) $\hat\beta_0 \approx -3.06$, $\hat\beta_1 \approx 2.16$ – similar to the coefficients of Exercise 1a.

b) The predicted probability is ≈ 5 % without a buried person, ≈ 29 % with one, ≈ 78 % with two and ≈ 97 % with three
buried persons. The S-curve rises steeply between one and two buried persons. Compared to the observed shares (≈ 2 %,
43 %, 52 %, 72 %) the curve is too low for one and too high for two and more buried persons: one linear score cannot
reproduce the observed pattern exactly – survival also depends on how quickly the buried persons are located and dug
out.

## Exercise 4: Which information is available when?

For a model to be useful, it may only use information that is **available at the time of the prediction**. We compare
two feature sets:

| feature set | features | situation |
|---|---|---|
| **A – terrain and bulletin** | `danger_level`, `elevation`, `aspect`, `activity`, `month`, `caught` | what is known about the place, the conditions and the group – but not how many were buried |
| **B – alarm with burial information** | set A + `fully_buried` | the alarm call reports how many persons are completely buried |

(`dead` must of course never be used – it *is* the outcome.)

a) Keep only accidents without missing values in these features. Create dummy variables for `aspect` and `activity`
(`pd.get_dummies(..., drop_first=True, dtype=int)`) and split the data into a stratified training (80 %) and test set
(20 %) with `random_state=42`.

b) Fit a logistic regression (with `StandardScaler`, in a `make_pipeline`) for both feature sets.

c) Plot the coefficients of model B. Which features increase the probability of a fatal outcome?

In [ ]:
# a) Complete cases, dummy variables and stratified split
features_a = ['danger_level', 'elevation', 'aspect', 'activity', 'month', 'caught']
features_b = features_a + ['fully_buried']
data = aval.dropna(subset=features_b).reset_index(drop=True)
X = pd.get_dummies(data[features_b], columns=['aspect', 'activity'], drop_first=True, dtype=int)
y = data['fatal']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=42)
print('Complete cases:', len(data), ' share fatal:', round(y.mean(), 3))
print('Training set:', X_train.shape, ' Test set:', X_test.shape)

cols_b = list(X.columns)
cols_a = [c for c in cols_b if c != 'fully_buried']

# b) Logistic regression for both feature sets
model_a = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X_train[cols_a], y_train)
model_b = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X_train[cols_b], y_train)

# c) Coefficients of model B
coefs = pd.Series(model_b[-1].coef_[0], index=cols_b).sort_values()
plt.figure(figsize=(7, 5))
coefs.plot(kind='barh', color=np.where(coefs > 0, 'darkred', 'steelblue'))
plt.axvline(0, color='black', linewidth=1)
plt.xlabel('Coefficient (standardised features)')
plt.title('Model B: coefficients')
plt.show()

**Solution:**

a) 2'170 complete accidents remain (13 % fatal); 1'736 are used for training, 434 for testing.

c) `fully_buried` has by far the largest coefficient (≈ +1.8 on the standardised scale): each completely buried person
strongly increases the probability of a fatal outcome. Interestingly, `caught` has a clearly **negative** coefficient
(≈ −0.6): *for the same number of completely buried persons*, more persons caught means that more people were only
partly buried or carried along and could start the companion rescue immediately. All other coefficients (danger level,
elevation, aspect, activity, month) are small.

## Exercise 5: Evaluation with the default threshold τ = 0.5

a) Predict the test set with model B and `predict()` (= threshold 0.5). Plot the confusion matrix and compute accuracy,
precision, recall, F1 and FPR.

b) Compare the accuracy with the baseline "never fatal". Is the model useful for the rescue coordination?

In [ ]:
# a) Confusion matrix and metrics
y_pred = model_b.predict(X_test[cols_b])
cm = confusion_matrix(y_test, y_pred, labels=[1, 0])
TN, FP, FN, TP = confusion_matrix(y_test, y_pred).ravel()

plt.figure(figsize=(4, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['fatal', 'not fatal'], yticklabels=['fatal', 'not fatal'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Model B, tau = 0.5')
plt.show()

print(f'Accuracy  = {accuracy_score(y_test, y_pred):.3f}')
print(f'Precision = {precision_score(y_test, y_pred):.3f}')
print(f'Recall    = {recall_score(y_test, y_pred):.3f}')
print(f'F1        = {f1_score(y_test, y_pred):.3f}')
print(f'FPR       = {FP / (FP + TN):.3f}')

# b) Baseline
print(f'\nBaseline accuracy (never fatal) = {1 - y_test.mean():.3f}')

**Solution:**

With $\tau = 0.5$ the model reaches an accuracy of **0.873** – practically the same as the baseline "never fatal"
(0.869)! It flags only 16 accidents as fatal, and the **recall is only 0.16**: 48 of 57 fatal accidents are missed
(precision 0.56, FPR 0.02). For the rescue coordination this is useless – the threshold 0.5 is far too high for a rare
and severe outcome. A model can be good at *ranking* (see AUC in Exercise 7) and still look bad with the default
threshold.

## Exercise 6: Choosing the threshold based on costs

The rescue coordination estimates:
- **underestimating** a fatal accident (FN: no doctor and no maximum alarm although a person is in mortal danger) is
  about **10 times** as bad as
- **overestimating** a non-fatal accident (FP: maximum alarm and additional resources that turn out to be unnecessary).

a) Compute precision, recall, FPR and the total costs $10\cdot FN + 1\cdot FP$ on the test set for the thresholds
$\tau = 0.05, 0.10, \dots, 0.90$ (use `predict_proba` of model B). Show the results as a table and plot the costs
against $\tau$.

b) Which threshold minimises the costs? How do recall and FPR change compared to $\tau = 0.5$?

*Note: strictly speaking, the threshold should be chosen on validation data (or with CV); here we use the test set to keep
the exercise short.*

In [ ]:
y_prob_b = model_b.predict_proba(X_test[cols_b])[:, 1]
cost_fn, cost_fp = 10, 1

rows = []
for tau in np.arange(0.05, 0.91, 0.05):
    y_pred_tau = (y_prob_b >= tau).astype(int)
    TN, FP, FN, TP = confusion_matrix(y_test, y_pred_tau).ravel()
    rows.append({'tau': round(tau, 2), 'TP': TP, 'FP': FP, 'FN': FN, 'TN': TN,
                 'precision': precision_score(y_test, y_pred_tau, zero_division=0),
                 'recall': TP / (TP + FN), 'FPR': FP / (FP + TN),
                 'costs': cost_fn * FN + cost_fp * FP})
thresholds = pd.DataFrame(rows)
print(thresholds.round(3).to_string(index=False))

best = thresholds.loc[thresholds['costs'].idxmin()]
print(f"\nCost-optimal threshold: tau = {best['tau']}, costs = {best['costs']:.0f}")
print(f"Costs at tau = 0.5:      {thresholds.loc[np.isclose(thresholds['tau'], 0.5), 'costs'].iloc[0]:.0f}")

plt.figure(figsize=(7, 4))
plt.plot(thresholds['tau'], thresholds['costs'], marker='o')
plt.axvline(best['tau'], color='darkred', linestyle='--', label=f"minimum at tau = {best['tau']}")
plt.xlabel('Threshold tau')
plt.ylabel('Total costs on the test set')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

**Solution:**

The cost-optimal threshold is **τ = 0.15** (0.10 is almost equally good). The recall rises from 0.16 to **0.88**
(50 of 57 fatal accidents are recognised), while the FPR rises from 0.02 to 0.14 and the precision drops slightly
(≈ 0.50). The total costs decrease from 487 to 121 (−75 %).

Rule of thumb: if FN are $c$ times as expensive as FP, the optimal threshold of a well-calibrated model is about
$\tau^* = 1/(1+c) = 1/11 \approx 0.09$ – close to the empirical optimum.

## Exercise 7: ROC curve and AUC – model A vs. model B

a) Plot the ROC curves of model A and model B on the test set and compute their AUC. Mark the points for $\tau = 0.5$
and for the cost-optimal threshold of model B.

b) Check the interpretation of the AUC empirically for model B: draw 100'000 random pairs (one fatal, one non-fatal
accident) from the test set and compute the share of pairs in which the fatal accident has the higher predicted
probability (ties count ½).

c) Interpret the AUC values. What does the comparison of model A and model B tell us?

In [ ]:
# a) ROC curves
y_prob_a = model_a.predict_proba(X_test[cols_a])[:, 1]
fpr_a, tpr_a, _ = roc_curve(y_test, y_prob_a)
fpr_b, tpr_b, _ = roc_curve(y_test, y_prob_b)
auc_a, auc_b = roc_auc_score(y_test, y_prob_a), roc_auc_score(y_test, y_prob_b)

plt.figure(figsize=(6, 6))
plt.plot(fpr_b, tpr_b, linewidth=2, label=f'B: with burial information (AUC = {auc_b:.3f})')
plt.plot(fpr_a, tpr_a, linewidth=2, label=f'A: terrain and bulletin only (AUC = {auc_a:.3f})')
plt.plot([0, 1], [0, 1], color='grey', linestyle='--', label='random ranking')
for tau, marker in [(0.5, 'o'), (best['tau'], 's')]:
    row = thresholds.loc[np.isclose(thresholds['tau'], tau)].iloc[0]
    plt.scatter(row['FPR'], row['recall'], s=80, marker=marker, color='black', zorder=3,
                label=f'B, tau = {tau}: TPR = {row["recall"]:.2f}, FPR = {row["FPR"]:.2f}')
plt.xlabel('False positive rate (FPR)')
plt.ylabel('True positive rate (recall)')
plt.title('ROC curves (test set)')
plt.legend(loc='lower right', fontsize=8)
plt.grid(alpha=0.3)
plt.show()

# b) AUC as probability of a correct ranking
rng = np.random.default_rng(42)
p_pos = y_prob_b[y_test.values == 1]
p_neg = y_prob_b[y_test.values == 0]
a = rng.choice(p_pos, 100_000)
b_ = rng.choice(p_neg, 100_000)
share = np.mean(a > b_) + 0.5 * np.mean(a == b_)
print(f'Share of correctly ranked pairs: {share:.3f}  vs. AUC = {auc_b:.3f}')

**Solution:**

a) Model B reaches an AUC of **≈ 0.90**, model A only **≈ 0.58** – barely better than random ranking (0.5). Moving the
threshold of model B from 0.5 to 0.15 moves us along the ROC curve to the upper right: much higher recall, somewhat
higher FPR. The AUC itself does not depend on the threshold.

b) The share of correctly ranked pairs (≈ 0.90) agrees with the AUC (up to sampling error).

c) In about 90 % of all (fatal, non-fatal) pairs, model B gives the fatal accident the higher probability. The
comparison of the two models shows that **whether an accident is fatal is decided almost entirely by the burial** – not
by terrain or bulletin. For the rescue coordination this means: the question "how many are completely buried?" is the
key information of the alarm call. For *prevention* the conclusion is different: terrain and danger level determine
whether an accident happens at all – but that question cannot be answered with accident data alone (we would also need
the tours without accident).

### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [ ]:
import os
import platform
import socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')